# 01 — Data Preprocessing and RFM Feature Engineering

**Owner: Member A**

This notebook turns `orders.csv` (and optionally `customers.csv`) into user-level RFM features and a repurchase label.

- Observation window: through **2024-12-31**.
- Prediction window: **2025-01-01 to 2025-12-31**.
- `Recency`: days from the observation cutoff to the user's most recent observed purchase.
- `Frequency`: number of observed orders.
- `Monetary`: total observed order value in USD.
- `Repurchase_Label`: 1 if the user has at least one valid order in the prediction window, otherwise 0.
- Output split: `train_70.csv`, `val_15.csv`, `test_15.csv`, plus `member_b_features.csv`.

**Important:** split is by unique customer after constructing one row per customer. Stratification is used where feasible. This notebook does not fit a scaler or K-Means; those are fitted only in Notebook 02 on the training split.

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
SEED = 42
ROOT = Path(".")
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OBSERVATION_END = pd.Timestamp("2024-12-31")
PREDICTION_START = pd.Timestamp("2025-01-01")
PREDICTION_END = pd.Timestamp("2025-12-31")

ORDERS_PATH = ROOT / "orders.csv"
CUSTOMERS_PATH = ROOT / "customers.csv"  # Optional; currently used for input validation only.

if not ORDERS_PATH.exists():
    raise FileNotFoundError("orders.csv was not found. Place it in the same folder as this notebook.")

orders = pd.read_csv(ORDERS_PATH)
print("Orders shape:", orders.shape)
print("Orders columns:", orders.columns.tolist())
display(orders.head())

if CUSTOMERS_PATH.exists():
    customers = pd.read_csv(CUSTOMERS_PATH)
    print("Customers shape:", customers.shape)
    print("Customers columns:", customers.columns.tolist())
else:
    customers = None
    print("customers.csv not found; proceeding using customer IDs in orders.csv.")

## 2. Validate and clean transaction data

In [ ]:
required = ["customer_id", "order_date", "order_id", "total_amount_usd"]
missing = [c for c in required if c not in orders.columns]
if missing:
    raise ValueError(f"orders.csv is missing required columns: {missing}")

orders_clean = orders.copy()
orders_clean["order_date"] = pd.to_datetime(orders_clean["order_date"], errors="coerce")
orders_clean["total_amount_usd"] = pd.to_numeric(orders_clean["total_amount_usd"], errors="coerce")

before = len(orders_clean)
orders_clean = orders_clean.dropna(subset=["customer_id", "order_date", "order_id", "total_amount_usd"])

# Source data includes a returned flag. Keep only non-returned orders when the column exists.
if "returned" in orders_clean.columns:
    orders_clean["returned"] = pd.to_numeric(orders_clean["returned"], errors="coerce")
    orders_clean = orders_clean[orders_clean["returned"] == 0]

# Exclude impossible dates outside the defined observation/prediction horizon.
orders_clean = orders_clean[
    (orders_clean["order_date"] <= PREDICTION_END)
].copy()

print(f"Rows before cleaning: {before:,}")
print(f"Rows after cleaning:  {len(orders_clean):,}")
print("Dropped rows:", before - len(orders_clean))
print("Missing customer IDs after cleaning:", int(orders_clean["customer_id"].isna().sum()))
print("Date range:", orders_clean["order_date"].min(), "to", orders_clean["order_date"].max())
display(orders_clean.head())

## 3. Construct RFM features and repurchase labels

In [ ]:
observation = orders_clean[orders_clean["order_date"] <= OBSERVATION_END].copy()
prediction = orders_clean[
    (orders_clean["order_date"] >= PREDICTION_START) &
    (orders_clean["order_date"] <= PREDICTION_END)
].copy()

# One row per customer in the observation window.
rfm = observation.groupby("customer_id").agg(
    last_purchase_date=("order_date", "max"),
    Frequency=("order_id", "nunique"),
    Monetary=("total_amount_usd", "sum")
)
rfm["Recency"] = (OBSERVATION_END - rfm["last_purchase_date"]).dt.days
rfm = rfm.drop(columns=["last_purchase_date"])

# A user is positive if they make at least one valid purchase in the prediction window.
repurchase_ids = set(prediction["customer_id"].unique())
rfm["Repurchase_Label"] = rfm.index.isin(repurchase_ids).astype(int)
rfm.index.name = "customer_id"
rfm = rfm.reset_index()

# Validate feature values.
rfm["Frequency"] = pd.to_numeric(rfm["Frequency"], errors="raise")
rfm["Monetary"] = pd.to_numeric(rfm["Monetary"], errors="raise")
rfm["Recency"] = pd.to_numeric(rfm["Recency"], errors="raise")
rfm = rfm.replace([np.inf, -np.inf], np.nan).dropna(subset=["Recency", "Frequency", "Monetary"])
rfm = rfm[(rfm["Frequency"] >= 1) & (rfm["Monetary"].notna())].copy()
rfm["Repurchase_Label"] = rfm["Repurchase_Label"].astype(int)

print("RFM table shape:", rfm.shape)
print("Unique customers:", rfm["customer_id"].nunique())
print("Target distribution:")
display(rfm["Repurchase_Label"].value_counts().rename(index={0:"No repurchase",1:"Repurchase"}).to_frame("count"))
print("Positive rate:", round(rfm["Repurchase_Label"].mean(), 4))
display(rfm.head())
rfm.to_csv(OUTPUT_DIR / "member_b_features.csv", index=False)
rfm.to_csv(ROOT / "member_b_features.csv", index=False)

## 4. Split customers into train / validation / test (70 / 15 / 15)

In [ ]:
# Split one row per customer. Stratify to keep label rates similar when both classes have enough samples.
X = rfm.drop(columns=["Repurchase_Label"])
y = rfm["Repurchase_Label"]

def can_stratify(labels, test_size):
    counts = labels.value_counts()
    n_test = int(np.ceil(len(labels) * test_size))
    n_train = len(labels) - n_test
    return len(counts) == 2 and counts.min() >= 2 and n_test >= 2 and n_train >= 2

stratify_first = y if can_stratify(y, 0.30) else None
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=stratify_first
)

stratify_second = y_temp if can_stratify(y_temp, 0.50) else None
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=stratify_second
)

train = X_train.copy()
train["Repurchase_Label"] = y_train.values
val = X_val.copy()
val["Repurchase_Label"] = y_val.values
test = X_test.copy()
test["Repurchase_Label"] = y_test.values

# Explicit customer_id column is safer than relying on CSV index behavior downstream.
for name, frame in [("train_70.csv", train), ("val_15.csv", val), ("test_15.csv", test)]:
    frame.to_csv(ROOT / name, index=False)
    frame.to_csv(OUTPUT_DIR / name, index=False)

print("Train:", train.shape, f"({len(train)/len(rfm):.1%})")
print("Validation:", val.shape, f"({len(val)/len(rfm):.1%})")
print("Test:", test.shape, f"({len(test)/len(rfm):.1%})")
display(pd.DataFrame({
    "split":["Train","Validation","Test"],
    "rows":[len(train),len(val),len(test)],
    "share":[len(train)/len(rfm),len(val)/len(rfm),len(test)/len(rfm)],
    "repurchase_rate":[train.Repurchase_Label.mean(),val.Repurchase_Label.mean(),test.Repurchase_Label.mean()]
}))

## 5. Quality checks and data-leakage checks

In [ ]:
# Check split proportions, label distribution, customer uniqueness, and overlap.
splits = {"train": train, "validation": val, "test": test}
for name, frame in splits.items():
    assert "customer_id" in frame.columns
    assert frame["customer_id"].is_unique, f"Duplicate customer_id found in {name}"
    assert set(frame["Repurchase_Label"].unique()).issubset({0, 1})
    print(f"{name:10s} n={len(frame):5d}, positive_rate={frame['Repurchase_Label'].mean():.4f}")

train_ids, val_ids, test_ids = (set(train.customer_id), set(val.customer_id), set(test.customer_id))
overlaps = {
    "train_validation": len(train_ids & val_ids),
    "train_test": len(train_ids & test_ids),
    "validation_test": len(val_ids & test_ids),
}
print("Customer overlaps:", overlaps)
assert all(v == 0 for v in overlaps.values()), "Customer leakage detected across splits."

print("\nFeature missing values:")
display(rfm[["Recency","Frequency","Monetary","Repurchase_Label"]].isna().sum().to_frame("missing"))
print("All checks passed.")


## Outputs

- `train_70.csv`, `val_15.csv`, `test_15.csv`: customer-level split files.
- `member_b_features.csv`: complete RFM table with target.
- `output/`: copies of the split files and feature table.

**Handoff to Member B:** use the three split CSVs. Fit the scaler, PCA, and K-Means on the training data only; apply the fitted transforms to validation and test. Keep `customer_id` and `Repurchase_Label` out of clustering/model features.